In [44]:
import json
import jax.numpy as jnp
import sys
sys.path.append('../../')

from pqcqec.noise.simple_noise import PennylaneNoisyGates

In [45]:
# data_file = '../json_data/old_data/1_5q_10g_5k_token_dict.json'
# with open(data_file, 'r') as f:
#     data_dict = json.load(f)
data_path = '../../nogit/circuit_tokens/no_uncomp/5q_10g_circuit_data/'
token_path = data_path + 'circuit_tokens.json'
config_path = data_path + 'config.json'
with open(token_path, 'r') as f:
    token_dict = json.load(f)

with open(config_path, 'r') as f:
    config = json.load(f)

num_qubits = config['qubits'][0]
num_gates = config['gates'][0]

base_ops = token_dict[0]['base_circuit_tokens']

print("Base Circuit Tokens ({}):".format(len(base_ops)))
for op in base_ops:
    print(f"{op}")




Base Circuit Tokens (10):
['cz', [1, 4], []]
['cx', [2, 1], []]
['h', [0], []]
['z', [4], []]
['h', [2], []]
['h', [4], []]
['cx', [4, 1], []]
['z', [2], []]
['h', [0], []]
['h', [0], []]


In [46]:
from pqcqec.simulate.simulate import run_circuit_with_noise_model
import jax.numpy as jnp

noise_model_noise = PennylaneNoisyGates(x_rad=0.0314, z_rad=0, delta_x=0, delta_z=0)
noise_model_ideal = PennylaneNoisyGates(x_rad=0, z_rad=0, delta_x=0, delta_z=0)

ZERO_STATE = jnp.zeros((2 ** num_qubits,), dtype=jnp.complex64).at[0].set(1.0)
ZERO_STATE

Array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j,
       0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j,
       0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j,
       0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],      dtype=complex64)

In [47]:
ideal_out_state = run_circuit_with_noise_model(base_ops, ZERO_STATE, noise_model_ideal, num_qubits)

In [48]:
measured_noPQC = run_circuit_with_noise_model(base_ops, ZERO_STATE, noise_model_noise, num_qubits)


In [49]:
from pqcqec.training.jax_loss_functions import jax_pure_state_fidelity

fidelity_noPQC = jax_pure_state_fidelity(ideal_out_state, measured_noPQC)

print(f"Fidelity (No PQC): {fidelity_noPQC:.4e}")


Fidelity (No PQC): 9.9557e-01


In [50]:
import os

NUM_QUBITS = config_dict['qubits'][0]
TOTAL_DATA = int(config_dict['seed'])

for seed in range(TOTAL_DATA):
    file = f"{seed}.json"
    if not os.path.exists(token_path + file):
        continue
    with open(token_path + file, 'r') as f:
        token_dict = json.load(f)

    base_ops = token_dict['base_circuit_tokens']
    pqc_ops = token_dict['pqc_circuit_tokens']
    
    measured_noPQC = run_circuit_with_noise_model(base_ops, ZERO_STATE, noise_model_noise, NUM_QUBITS)
    measured_PQC = run_circuit_with_noise_model(pqc_ops, ZERO_STATE, noise_model_noise, NUM_QUBITS)

    fidelity_noPQC = jax_pure_state_fidelity(ZERO_STATE, measured_noPQC)
    fidelity_PQC = jax_pure_state_fidelity(ZERO_STATE, measured_PQC)

    print(f"For {seed} : Fidelity (No PQC): {fidelity_noPQC:.4e}, Fidelity (PQC): {fidelity_PQC:.4e}")


NameError: name 'config_dict' is not defined